# Process cell ranger output

## Step 1: Loading the Data  First, let's load the dataset into a pandas DataFrame.

### Step 1.1: Get the file location

In [ ]:
## file folder
import os

folder_path = '/storage/users/data/PANC/cellranger_outs/'

# Get the list of files and directories in the folder
folder_content = os.listdir(folder_path)

# Print the content of the folder
for item in folder_content:
    print(item)

In [ ]:
## print head of example file
import csv

def print_csv_head(file_path, num_rows):
    with open(file_path, 'r') as file:
        csv_reader = csv.reader(file)
        header = next(csv_reader)  # Read the header
        print(','.join(header))  # Print the header
        
        # Print the specified number of rows
        for _ in range(num_rows):
            row = next(csv_reader, None)
            if row is None:
                break
            print(','.join(row))

file_path = '/storage/users/data/PANC/CTRL_1/metrics_summary.csv'
num_rows = 5  # Specify the number of rows to print

print_csv_head(file_path, num_rows)

In [ ]:
# Load the gene expression matrices for each sample
print(folder_content)


### Step 1.2: Unzip the data

In [ ]:
import os
import gzip


In [ ]:
def unzip_all_gz_files(folder_path):
    for file_name in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file_name)
        

        if file_name.endswith('.gz'):
            unzipped_file_path = os.path.splitext(file_path)[0]  # Remove the .gz extension
            #print(file_path)
            with gzip.open(file_path, 'rb') as gz_file:
                with open(unzipped_file_path, 'wb') as unzipped_file:
                    unzipped_file.write(gz_file.read())
            
            # Optionally, you can delete the .gz file after unzipping
            # os.remove(file_path)


samples = folder_content
# or
samples = ['CTRL_1', 'CTRL_2', 'CTRL_2D', 'GEM_2', 'TGFb1_1', 'TGFb1_2', 'TGFb1_GEM_1', 'TGFb1_GEM_2']
for sample in samples:

    folder_path = f'/storage/users/data/PANC/{sample}/count/sample_filtered_feature_bc_matrix'
    df = unzip_all_gz_files(folder_path)

    #print(df.head())
    print(folder_path)
    

### Step 1.3: Load the files example  https://support.10xgenomics.com/single-cell-gene-expression/software/pipelines/latest/output/matrices

In [ ]:
from tqdm import tqdm
import pandas as pd

In [ ]:
import scipy.io as sio
import os

In [ ]:
import anndata as ad

In [ ]:
import scanpy as sc

In [ ]:
!conda list

In [ ]:
#adata = ad.AnnData()
adata_list = []

In [ ]:
combined_adata = ad.AnnData()

In [ ]:
#samples = ['sample1', 'sample2']  # Add more sample names if needed
conditions = folder_content
conditions = samples
#pwd /storage/users/sac43cg/res_Samantha/outs/per_sample_outs/CTRL_1/count/sample_filtered_feature_bc_matrix

# Create an empty list to store the AnnData objects
adata_list = []
barcodes_list = []
features_list = []
matrix_list = []
#combined_adata = []

for condition in conditions:
           
    # Define the file paths
    base_path = f'/storage/users/data/PANC/{condition}/count/sample_filtered_feature_bc_matrix/'  # Replace with the actual path to your data

    barcodes_file = os.path.join(base_path, 'barcodes.tsv')
    features_file = os.path.join(base_path, 'features.tsv')
    matrix_file = os.path.join(base_path, 'matrix.mtx')

    # Read the barcodes.tsv file
    barcodes = pd.read_csv(barcodes_file, header=None, index_col=0, names=['barcode'])

    # Read the features.tsv file
    features = pd.read_csv(features_file, sep='\t', header=None, index_col=0, names=['feature_id', 'symbol', 'type'])
    # get only gene expression features
    #features = features[features['type'] == 'gene expression']

    # Read the matrix.mtx file
    matrix = sio.mmread(matrix_file)

    # Create the AnnData object
    adata = ad.AnnData(X=matrix.T, obs=barcodes, var=features)
    
    # Add a column to the observation metadata to indicate the condition
    adata.obs['condition'] = condition
    #adata.varm['ensembl'] = features['feature_id']
    #adata.varp['symbol'] = features['symbol']
    #adata.varp['type'] = features['type']
    #adata.varm['condition'] = condition

    # Optional: If you have additional annotations for cells or features, you can add them to the AnnData object as well.
    # For example, if you have a file called 'annotations.csv' with additional cell annotations:
    #annotations_file = os.path.join(base_path, 'annotations.csv')
    #annotations = pd.read_csv(annotations_file, index_col=0)
    # Add cell annotations
    #adata.obs = adata.obs.merge(annotations, left_index=True, right_index=True)

    # Append the AnnData object to the list
    adata_list.append(adata)
    features['condition'] = condition
    features_list.append(features)
        
    
# Combine all the AnnData objects into a single AnnData object
combined_adata = ad.concat(adata_list, join='outer')

# Rename duplicate variable and observation names
#combined_adata.var_names_make_unique()
#combined_adata.obs_names_make_unique()




In [ ]:
combined_adata.write('/storage/users/data/PANC/combined_adata.h5ad')

### Step 1.3.1: Check the new adata (.h5ad) file


In [ ]:
# Print the annotations
print("Observation annotations:")
print(combined_adata.obs.head())  # Print the first few rows of observation annotations

print("\nVariable annotations:")
print(combined_adata.var.head())  # Print the first few rows of variable annotations

print("\nVariable annotations:")
print(combined_adata.varm)  # Print the first few rows of variable annotations

print("\nVariable features:")
print(features.head())
print(features['type'].unique())

# Print the dimensions
print("\nNumber of observations (cells):", combined_adata.n_obs)
print("Number of variables (features):", combined_adata.n_vars)



In [ ]:
# Iterate over each condition
for condition in combined_adata.obs['condition'].unique():
    condition_adata = combined_adata[combined_adata.obs['condition'] == condition]
    
    # Get the number of features and barcodes for the current condition
    num_features = condition_adata.n_vars
    num_barcodes = condition_adata.n_obs
    
    print("Observation annotations:")
    print(condition_adata.obs.head())  # Print the first few rows of observation annotations
    
    print("\nVariable annotations:")
    print(condition_adata.var.head())  # Print the first few rows of variable annotations
    
    # Write the condition name, number of features, and number of barcodes to the file
    print(f"Condition: {condition}\n")
    print(f"Number of features: {num_features}\n")
    print(f"Number of barcodes: {num_barcodes}\n\n")

# Close the file

In [ ]:
print(type(barcodes))
print(type(features))
print(type(adata))
print('\n')


In [ ]:
dimensions = barcodes.shape
print('\n' , dimensions)
dimensions = features.shape
print('\n' , dimensions)
dimensions = combined_adata.shape
print('\n' , dimensions)
dimensions = matrix.shape
print('\n' , dimensions)

### end of loading